# Apache Iceberg com Apache Spark

Este notebook demonstra as operações fundamentais do **Apache Iceberg** com **PySpark**:
- Configuração do SparkSession com Iceberg
- Criação do catálogo e tabela
- Carga inicial de dados (INSERT)
- Atualização de registros (UPDATE)
- Remoção de registros (DELETE)
- Gerenciamento de Snapshots

**Dataset:** Vendas de E-commerce

> **Nota:** A primeira execução pode demorar alguns minutos pois o Spark irá baixar o JAR do Iceberg automaticamente via Maven.

## 1. Configuração do SparkSession com Apache Iceberg

In [ ]:
import os

# Configura o pacote Iceberg para download via Maven antes de criar o SparkSession
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.1 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import col, lit

spark = SparkSession.builder \
    .appName("Apache Iceberg Demo") \
    .master("local[*]") \
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    ) \
    .config(
        "spark.sql.catalog.local",
        "org.apache.iceberg.spark.SparkCatalog"
    ) \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "./iceberg-warehouse") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("✅ SparkSession com Iceberg criada com sucesso!")
print(f"   Versão do Spark : {spark.version}")
print(f"   Master          : {spark.sparkContext.master}")

## 2. Criação do Namespace e Tabela Iceberg

In [ ]:
# Cria o namespace (database) no catálogo local
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.db")

# Remove a tabela se existir (para reexecução do notebook)
spark.sql("DROP TABLE IF EXISTS local.db.vendas")

# Cria a tabela Iceberg
spark.sql("""
    CREATE TABLE local.db.vendas (
        id         INT,
        produto    STRING,
        categoria  STRING,
        quantidade INT,
        preco      DOUBLE,
        data_venda STRING,
        vendedor   STRING,
        status     STRING
    )
    USING iceberg
    TBLPROPERTIES (
        'write.format.default' = 'parquet',
        'write.parquet.compression-codec' = 'snappy'
    )
""")

print("✅ Namespace 'local.db' e tabela 'vendas' criados com sucesso!")
spark.sql("DESCRIBE TABLE local.db.vendas").show(truncate=False)

## 3. Definição do Schema e Dataset

In [ ]:
schema = StructType([
    StructField("id",         IntegerType(), False),
    StructField("produto",    StringType(),  True),
    StructField("categoria",  StringType(),  True),
    StructField("quantidade", IntegerType(), True),
    StructField("preco",      DoubleType(),  True),
    StructField("data_venda", StringType(),  True),
    StructField("vendedor",   StringType(),  True),
    StructField("status",     StringType(),  True),
])

dados_iniciais = [
    (1,  "Notebook Dell",    "Eletrônicos", 2, 3500.00, "2024-01-15", "Ana Silva",    "entregue"),
    (2,  "Camiseta Polo",    "Roupas",      5,   89.90, "2024-01-20", "Bruno Costa",  "entregue"),
    (3,  "Tênis Running",    "Calçados",    1,  450.00, "2024-02-01", "Carlos Lima",  "pago"),
    (4,  "Mouse Gamer",      "Eletrônicos", 3,  250.00, "2024-02-10", "Diana Ramos",  "entregue"),
    (5,  "Calça Jeans",      "Roupas",      2,  199.90, "2024-02-14", "Eduardo Melo", "entregue"),
    (6,  "Smartphone X",     "Eletrônicos", 1, 2800.00, "2024-02-20", "Ana Silva",    "pago"),
    (7,  "Mochila Viagem",   "Acessórios",  1,  350.00, "2024-03-01", "Bruno Costa",  "pendente"),
    (8,  "Teclado Mecânico", "Eletrônicos", 1,  450.00, "2024-03-05", "Carlos Lima",  "pago"),
    (9,  "Vestido Floral",   "Roupas",      3,  159.90, "2024-03-10", "Diana Ramos",  "entregue"),
    (10, "Monitor 27'",      "Eletrônicos", 1, 1800.00, "2024-03-15", "Eduardo Melo", "pendente"),
    (11, "Sandália Couro",   "Calçados",    2,  280.00, "2024-03-18", "Ana Silva",    "entregue"),
    (12, "Câmera DSLR",      "Eletrônicos", 1, 4200.00, "2024-03-20", "Bruno Costa",  "pago"),
    (13, "Jaqueta Denim",    "Roupas",      1,  320.00, "2024-03-22", "Carlos Lima",  "cancelado"),
    (14, "Headset USB",      "Eletrônicos", 2,  180.00, "2024-03-25", "Diana Ramos",  "cancelado"),
    (15, "Relógio Analógico","Acessórios",  1,  520.00, "2024-03-28", "Eduardo Melo", "pendente"),
]

print(f"✅ Schema e dataset definidos.")
print(f"   Total de registros iniciais: {len(dados_iniciais)}")

## 4. INSERT — Carga Inicial

In [ ]:
df_inicial = spark.createDataFrame(dados_iniciais, schema)

# Insere os dados na tabela Iceberg
df_inicial.writeTo("local.db.vendas").append()

print("✅ Dados inseridos na tabela Iceberg!")
resultado = spark.sql("SELECT COUNT(*) AS total FROM local.db.vendas")
resultado.show()

spark.sql("SELECT * FROM local.db.vendas").show(truncate=False)

### 4.1 INSERT — Novos Pedidos

In [ ]:
# INSERT via SQL
spark.sql("""
    INSERT INTO local.db.vendas VALUES
    (16, 'Fone Bluetooth',  'Eletrônicos', 1, 299.90, '2024-04-01', 'Ana Silva',    'pendente'),
    (17, 'Bolsa de Couro',  'Acessórios',  1, 480.00, '2024-04-02', 'Bruno Costa',  'pendente'),
    (18, 'Tênis Social',    'Calçados',    1, 390.00, '2024-04-03', 'Carlos Lima',  'pendente')
""")

total = spark.sql("SELECT COUNT(*) AS total FROM local.db.vendas").collect()[0][0]
print(f"✅ Novos registros inseridos via SQL!")
print(f"   Total de registros agora: {total}")

spark.sql("SELECT * FROM local.db.vendas WHERE id >= 16").show(truncate=False)

## 5. UPDATE — Atualizando Registros

O Iceberg suporta UPDATE com SQL a partir da versão 0.14.

In [ ]:
print("--- Antes do UPDATE: registros pendentes ---")
spark.sql("""
    SELECT id, produto, vendedor, status
    FROM local.db.vendas
    WHERE status = 'pendente'
""").show(truncate=False)

# UPDATE: pedidos 'pendente' → 'pago'
spark.sql("""
    UPDATE local.db.vendas
    SET status = 'pago'
    WHERE status = 'pendente'
""")

print("--- Após UPDATE (pendente → pago) ---")
spark.sql("""
    SELECT id, produto, vendedor, status
    FROM local.db.vendas
    WHERE id IN (7, 10, 15, 16, 17, 18)
""").show(truncate=False)

In [ ]:
# UPDATE 2: pagos antigos → entregue
spark.sql("""
    UPDATE local.db.vendas
    SET status = 'entregue'
    WHERE status = 'pago' AND data_venda < '2024-03-01'
""")

print("✅ UPDATE 2: pagos antigos → entregue")
spark.sql("""
    SELECT status, COUNT(*) AS total
    FROM local.db.vendas
    GROUP BY status
    ORDER BY status
""").show()

## 6. DELETE — Removendo Registros

In [ ]:
print("--- Registros a serem deletados (status = 'cancelado') ---")
spark.sql("""
    SELECT * FROM local.db.vendas WHERE status = 'cancelado'
""").show(truncate=False)

spark.sql("""
    DELETE FROM local.db.vendas WHERE status = 'cancelado'
""")

total_pos = spark.sql("SELECT COUNT(*) AS total FROM local.db.vendas").collect()[0][0]
print(f"✅ DELETE executado!")
print(f"   Registros restantes: {total_pos}")

spark.sql("""
    SELECT status, COUNT(*) AS total
    FROM local.db.vendas
    GROUP BY status ORDER BY status
""").show()

## 7. Gerenciamento de Snapshots e Time Travel

In [ ]:
# Listar todos os snapshots gerados
print("=== Snapshots da Tabela ===")
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM local.db.vendas.snapshots
""").show(truncate=False)

In [ ]:
# Histórico de operações
print("=== Histórico da Tabela ===")
spark.sql("""
    SELECT made_current_at, snapshot_id, is_current_ancestor
    FROM local.db.vendas.history
""").show(truncate=False)

In [ ]:
# Time Travel: ler o primeiro snapshot (dados originais)
primeiro_snapshot = spark.sql("""
    SELECT snapshot_id FROM local.db.vendas.snapshots ORDER BY committed_at ASC
""").collect()[0][0]

print(f"=== Time Travel: Snapshot original (ID: {primeiro_snapshot}) ===")
df_original = spark.read \
    .option("snapshot-id", primeiro_snapshot) \
    .table("local.db.vendas")

print(f"   Registros no snapshot original: {df_original.count()}")
cancelados_original = df_original.filter(col("status") == "cancelado").count()
cancelados_atual    = spark.sql("SELECT COUNT(*) FROM local.db.vendas WHERE status = 'cancelado'").collect()[0][0]

print(f"   Cancelados no snapshot original : {cancelados_original}")
print(f"   Cancelados na tabela atual      : {cancelados_atual} (foram deletados)")

## 8. Leitura Final e Estatísticas

In [ ]:
print("=== Estado Final da Tabela ===")
spark.sql("SELECT * FROM local.db.vendas ORDER BY id").show(truncate=False)

print("\n=== Estatísticas por Categoria ===")
spark.sql("""
    SELECT
        categoria,
        COUNT(id)                        AS total_pedidos,
        SUM(quantidade)                  AS total_itens,
        ROUND(SUM(preco * quantidade), 2) AS receita_total,
        ROUND(AVG(preco), 2)             AS preco_medio
    FROM local.db.vendas
    GROUP BY categoria
    ORDER BY receita_total DESC
""").show(truncate=False)

spark.stop()
print("\n✅ SparkSession encerrada.")